# Full-data MLD hyperparameter optimization

This notebook optimizes the **exact production MLD decoder** on a deterministic subset of the labelled training data. It does not use cross-validation folds. The objective is the total number of incorrect logical bits, with combined LER as the displayed metric and runtime as a tie-breaker.

The decoder path is identical to `submission_template/decoder.py`: forward/reverse MLD tables, `mld_logw`, uncertainty-based refinement, reversed detector columns for the reverse sweep, per-class `np.maximum` log-weight merging, `marginals_logw`, and a 0.5 logical-bit threshold.

The search has four stages:

1. Evaluate the current production baseline.
2. Perform bounded coordinate search on numeric parameters using full labelled data.
3. Test joint 3x3 neighborhoods around the coordinate optimum.
4. Test alternate sweep orders and an additional-pass structure.

This can be expensive on the full dataset. The notebook is intentionally configured to run the complete search when its evaluation cell is executed.


In [1]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("repository:", REPO_ROOT)
print("python:", sys.executable)


repository: /Users/suneg/Dev/QIS-hackathon/qis-my-error
python: /Users/suneg/Dev/QIS-hackathon/qis-my-error/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import importlib
import time
from dataclasses import dataclass
from itertools import product

import numpy as np
import stim

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info

# Reload the repository MLD module so a stale notebook import cannot change the API.
sys.modules.pop("mld", None)
mld = importlib.import_module("mld")
for required in ("build_mld", "marginals_logw", "mld_logw"):
    if not hasattr(mld, required):
        raise ImportError(f"loaded unexpected MLD module {getattr(mld, '__file__', None)}")
build_mld = mld.build_mld
marginals_logw = mld.marginals_logw
mld_logw = mld.mld_logw

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SEED = 42


/Users/suneg/Dev/QIS-hackathon/qis-my-error/submission_template/bp_osd.py:48: RuntimeWarning: overflow encountered in scalar multiply
  _CTZ_TABLE[((np.uint64(1) << np.uint64(_i)) * _DEBRUIJN) >> np.uint64(58)] = _i


## Load a deterministic labelled subset

To keep repeated MLD evaluations practical, this loads a fixed, reproducible subset from each basis. Increase `SUBSET_PER_BASIS` for a more precise final comparison after the search has narrowed the candidates.


In [3]:
SUBSET_PER_BASIS = 20000

def load_full_data(path: Path, subset_per_basis: int | None, seed: int):
    data = load_kaggle_csv(str(path))
    if "observable_flips" not in data:
        raise ValueError("full-data optimization requires observable_flips labels")
    result = {}
    for basis in ("X", "Z"):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no labelled {basis} rows found")
        rng = np.random.default_rng(seed + (0 if basis == "X" else 1))
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": np.ascontiguousarray(data["detector_events"][rows], dtype=np.uint8),
            "labels": np.ascontiguousarray(data["observable_flips"][rows], dtype=bool),
        }
    return result


def load_problem_data(folder: Path):
    result = {}
    for basis in ("X", "Z"):
        circuit = stim.Circuit.from_file(str(folder / f"{basis}_circuit.stim"))
        result[basis] = problem_info(circuit)
    return result

basis_data = load_full_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
problems = load_problem_data(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(f"{basis}: shots={len(basis_data[basis]['events'])}, events={basis_data[basis]['events'].shape}, labels={basis_data[basis]['labels'].shape}")


X: shots=20000, events=(20000, 70), labels=(20000, 4)
Z: shots=20000, events=(20000, 70), labels=(20000, 4)


## Exact production-equivalent decoder and objective

`decode_candidate` is deliberately written to match `decoder.decode_marginals`. A candidate tier is `(direction, eps, max_states, threshold)`, where direction 0 uses normal detector order and direction 1 uses reversed detector order.


In [4]:
TABLE_CACHE = {}
EVALUATION_CACHE = {}


def get_tables(basis: str, hexp: float):
    key = (basis, float(hexp))
    if key not in TABLE_CACHE:
        info = problems[basis]
        TABLE_CACHE[key] = (
            build_mld(info["H"], info["L"], info["p"], hexp=hexp),
            build_mld(info["H"][::-1].copy(), info["L"], info["p"], hexp=hexp),
        )
    return TABLE_CACHE[key]


def decode_candidate(tables, detector_events, tiers, compact_every):
    D = np.ascontiguousarray(np.asarray(detector_events, dtype=bool))
    events = (D, np.ascontiguousarray(D[:, ::-1]))
    logw = None
    for direction, eps, max_states, threshold in tiers:
        if logw is None:
            logw = mld_logw(
                tables[direction], events[direction], eps=eps,
                max_states=int(max_states), compact_every=int(compact_every),
            )
            continue
        selected = np.flatnonzero(
            (np.abs(marginals_logw(logw) - 0.5) < 0.5 - threshold).any(axis=1)
        )
        if len(selected):
            refined = mld_logw(
                tables[direction], events[direction][selected], eps=eps,
                max_states=int(max_states), compact_every=int(compact_every),
            )
            logw[selected] = np.maximum(logw[selected], refined)
    return (marginals_logw(logw) > 0.5).astype(np.uint8)


def candidate_key(candidate):
    return (candidate["tiers"], float(candidate["hexp"]), int(candidate["compact_every"]))


def evaluate_candidate(candidate, verbose=True):
    key = candidate_key(candidate)
    if key in EVALUATION_CACHE:
        return EVALUATION_CACHE[key]
    started = time.perf_counter()
    total_wrong = 0
    total_bits = 0
    basis_scores = {}
    for basis in ("X", "Z"):
        tables = get_tables(basis, candidate["hexp"])
        predictions = decode_candidate(
            tables,
            basis_data[basis]["events"],
            candidate["tiers"],
            candidate["compact_every"],
        )
        labels = basis_data[basis]["labels"]
        wrong = int(np.count_nonzero(predictions.astype(bool) != labels))
        bits = int(predictions.size)
        total_wrong += wrong
        total_bits += bits
        basis_scores[basis] = {
            "wrong": wrong,
            "bits": bits,
            "ler": logical_error_rate(predictions, labels),
        }
    result = {
        "candidate": candidate,
        "wrong": total_wrong,
        "bits": total_bits,
        "combined_ler": total_wrong / total_bits,
        "seconds": time.perf_counter() - started,
        "basis": basis_scores,
    }
    EVALUATION_CACHE[key] = result
    if verbose:
        print(f"wrong={total_wrong} LER={result['combined_ler']:.8f} time={result['seconds']:.2f}s candidate={candidate}")
    return result


def objective_key(result):
    return (result["wrong"], result["seconds"])


## Baseline and bounded coordinate search

The baseline matches the current production configuration. Coordinate search evaluates both neighboring values for each numeric parameter, moves to an improving neighbor, and repeats until a full round makes no move or the safety limits are reached.


In [5]:
BASELINE = {
    "tiers": ((0, 6.666666666666667e-5, 1792, 0.0), (1, 3e-5, 3072, 0.1)),
    "hexp": 0.6,
    "compact_every": 32,
}
COORDINATE_MAX_ROUNDS = 30
COORDINATE_MAX_STEPS_PER_PARAMETER = 8


def point_from_candidate(candidate):
    first, second = candidate["tiers"]
    return {
        "first_eps": first[1], "first_states": first[2],
        "second_eps": second[1], "second_states": second[2],
        "threshold": second[3], "hexp": candidate["hexp"],
        "compact_every": candidate["compact_every"],
    }


def candidate_from_point(point, directions=(0, 1)):
    return {
        "tiers": (
            (directions[0], point["first_eps"], int(point["first_states"]), 0.0),
            (directions[1], point["second_eps"], int(point["second_states"]), point["threshold"]),
        ),
        "hexp": float(point["hexp"]),
        "compact_every": int(point["compact_every"]),
    }


def neighbors(point, name):
    lower, upper = point.copy(), point.copy()
    value = point[name]
    if name in ("first_eps", "second_eps"):
        lower[name] = max(1e-7, value / 1.5)
        upper[name] = min(1e-2, value * 1.5)
    elif name in ("first_states", "second_states"):
        lower[name] = max(128, value - 256)
        upper[name] = min(32768, value + 256)
    elif name in ("threshold", "hexp"):
        lower[name] = max(0.0, value - 0.025)
        upper[name] = min(0.95 if name == "threshold" else 1.5, value + 0.025)
    elif name == "compact_every":
        lower[name] = max(4, value - 8)
        upper[name] = min(128, value + 8)
    return lower, upper


def coordinate_search(start=BASELINE):
    point = point_from_candidate(start)
    current = evaluate_candidate(candidate_from_point(point))
    names = ("first_eps", "first_states", "second_eps", "second_states", "threshold", "hexp", "compact_every")
    history = []
    for round_index in range(1, COORDINATE_MAX_ROUNDS + 1):
        moved = False
        for name in names:
            for _ in range(COORDINATE_MAX_STEPS_PER_PARAMETER):
                lower, upper = neighbors(point, name)
                candidates = [(point, current)]
                if lower[name] != point[name]:
                    candidates.append((lower, evaluate_candidate(candidate_from_point(lower))))
                if upper[name] != point[name]:
                    candidates.append((upper, evaluate_candidate(candidate_from_point(upper))))
                best_point, best_result = min(candidates, key=lambda item: objective_key(item[1]))
                if objective_key(best_result) >= objective_key(current):
                    break
                old = point[name]
                point, current = best_point, best_result
                moved = True
                history.append((round_index, name, old, point[name], current["wrong"], current["combined_ler"]))
                print(f"move {name}: {old} -> {point[name]}, wrong={current['wrong']}")
        if not moved:
            print("coordinate search converged")
            break
    return candidate_from_point(point), current, history

coordinate_candidate, coordinate_result, coordinate_history = coordinate_search()
print("coordinate candidate:", coordinate_candidate)
print("coordinate result:", coordinate_result)


wrong=5368 LER=0.03355000 time=19.57s candidate={'tiers': ((0, 6.666666666666667e-05, 1792, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}
wrong=5365 LER=0.03353125 time=16.31s candidate={'tiers': ((0, 4.4444444444444447e-05, 1792, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}
wrong=5368 LER=0.03355000 time=12.18s candidate={'tiers': ((0, 0.0001, 1792, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}
move first_eps: 6.666666666666667e-05 -> 4.4444444444444447e-05, wrong=5365
wrong=5368 LER=0.03355000 time=19.05s candidate={'tiers': ((0, 2.962962962962963e-05, 1792, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}
wrong=5365 LER=0.03353125 time=16.61s candidate={'tiers': ((0, 4.4444444444444447e-05, 1536, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}
wrong=5365 LER=0.03353125 time=12.89s candidate={'tiers': ((0, 4.4444444444444447e-05, 2048, 0.0), (1, 3e-05, 3072, 0.1)), 'hexp': 0.6, 'compact_every': 32}


## Joint local refinement and decoder-structure search

Coordinate search can miss interactions between parameters. This stage evaluates 3x3 neighborhoods for the two epsilon/state pairs, then tests alternate direction orders and a three-pass structure.


In [6]:
def three_values(value, name):
    lower, upper = neighbors({name: value}, name)
    return (lower[name], value, upper[name])


def joint_candidates(base):
    point = point_from_candidate(base)
    for first_eps, first_states, second_eps, second_states in product(
        three_values(point["first_eps"], "first_eps"),
        three_values(point["first_states"], "first_states"),
        three_values(point["second_eps"], "second_eps"),
        three_values(point["second_states"], "second_states"),
    ):
        p = point.copy()
        p.update(first_eps=first_eps, first_states=first_states, second_eps=second_eps, second_states=second_states)
        yield candidate_from_point(p)

joint_results = []
for candidate in joint_candidates(coordinate_candidate):
    joint_results.append(evaluate_candidate(candidate))
joint_best = min(joint_results, key=objective_key)

structure_candidates = [
    joint_best["candidate"],
    {**joint_best["candidate"], "tiers": tuple(reversed(joint_best["candidate"]["tiers"]))},
]
first, second = joint_best["candidate"]["tiers"]
structure_candidates.extend([
    {
        **joint_best["candidate"],
        "tiers": (
            first,
            second,
            (0, second[1], second[2], 0.25),
        ),
    },
    {
        **joint_best["candidate"],
        "tiers": (
            second,
            first,
            (1, first[1], first[2], 0.25),
        ),
    },
])
structure_results = [evaluate_candidate(candidate) for candidate in structure_candidates]
final_result = min(structure_results, key=objective_key)

print("joint best:", joint_best)
print("final best candidate:", final_result["candidate"])
print("final best result:", final_result)


wrong=5365 LER=0.03353125 time=10.29s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 3e-05, 2560, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5365 LER=0.03353125 time=12.56s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 3e-05, 2816, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5365 LER=0.03353125 time=14.72s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 3e-05, 3072, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5370 LER=0.03356250 time=9.23s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 4.5e-05, 2560, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5370 LER=0.03356250 time=11.05s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 4.5e-05, 2816, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5371 LER=0.03356875 time=11.27s candidate={'tiers': ((0, 4.4444444444444447e-05, 2560, 0.0), (1, 4.5e-05, 3072, 0.125)), 'hexp': 0.6, 'compact_every': 32}
wrong=5373 LER=0.03358125 time=8.64s cand

## Final report

The result below is the best full-data empirical configuration found by this notebook. Re-run the final candidate on a fresh holdout before treating a very small improvement as robust.


In [7]:
print("BEST PRODUCTION PARAMETERS")
print("MLD_TIERS =", final_result["candidate"]["tiers"])
print("MLD_COMPACT_EVERY =", final_result["candidate"]["compact_every"])
print("MLD_HEXP =", final_result["candidate"]["hexp"])
print("wrong logical bits =", final_result["wrong"])
print("total logical bits =", final_result["bits"])
print("combined LER =", final_result["combined_ler"])
print("runtime seconds =", final_result["seconds"])
print("basis results =", final_result["basis"])
print("evaluated unique candidates =", len(EVALUATION_CACHE))


BEST PRODUCTION PARAMETERS
MLD_TIERS = ((0, 6.666666666666667e-05, 3072, 0.0), (1, 4.5e-05, 3072, 0.125))
MLD_COMPACT_EVERY = 32
MLD_HEXP = 0.6
wrong logical bits = 5363
total logical bits = 160000
combined LER = 0.03351875
runtime seconds = 6.548573499996564
basis results = {'X': {'wrong': 2718, 'bits': 80000, 'ler': 0.033975}, 'Z': {'wrong': 2645, 'bits': 80000, 'ler': 0.0330625}}
evaluated unique candidates = 131
